# Toxion: Advanced analysis

Endpoint dependence, chemical space, optimizers, imbalance-aware losses and calibration. Derivations are in `docs/math.md`.

Needs `uv sync --extra notebook --extra ml` (or `pip install -e ".[notebook,ml]"`). If the cleaned Tox21 file does not exist yet, the notebook falls back to **synthetic** molecules so it still runs. Nothing from synthetic data is a result.

The train/val/test split here is **random and only for demonstration**. A scaffold split should be used for production benchmarks; do not report numbers from this notebook.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
CLEAN = ROOT / "data/processed/tox21_clean.parquet"

from toxion.data.loader import TOX21_TASKS, load_clean

if CLEAN.exists():
    df = load_clean(str(CLEAN))
    SYNTHETIC = False
else:
    from toxion.data.synthetic import make_synthetic_clean
    df = make_synthetic_clean(600, seed=0)
    SYNTHETIC = True
    print("WARNING: using SYNTHETIC data. Run `toxion prepare-data` for the real thing.")
print(df.shape, "synthetic" if SYNTHETIC else "real")

## 1. Endpoint dependence
phi = Pearson correlation of two binary endpoints; mutual information in bits. Both use only molecules labeled for both endpoints.

In [ ]:
from toxion.analysis.endpoints import endpoint_dependence

dep = endpoint_dependence(df)

def heatmap(frame, title, cmap, vmin=None, vmax=None):
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(frame.to_numpy(), cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(frame)), frame.columns, rotation=70, ha="right")
    ax.set_yticks(range(len(frame)), frame.index)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()

heatmap(dep["phi"], "phi coefficient", "coolwarm", -1, 1)
heatmap(dep["mi_bits"], "mutual information (bits)", "viridis")

## 2. Chemical space
PCA by SVD of Morgan bit fingerprints, nearest-neighbor Tanimoto similarity, scaffold diversity.

In [ ]:
from toxion.analysis.chemspace import (
    internal_diversity, morgan_bits, nearest_neighbor_similarity, pca_svd, scaffold_stats,
)

smiles = df["smiles_canonical"].tolist()
N_BITS = 2048
bits = morgan_bits(smiles, n_bits=N_BITS)
pca = pca_svd(bits, n_components=20)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
k = np.arange(1, len(pca.explained_variance_ratio) + 1)
axes[0].bar(k, pca.explained_variance_ratio)
axes[0].plot(k, np.cumsum(pca.explained_variance_ratio), "k.-")
axes[0].set(title="PCA explained variance (bars) and cumulative (line)", xlabel="component")

ENDPOINT = "NR-AhR"
labels = df[ENDPOINT].to_numpy()
for value, color, name in [(np.nan, "lightgray", "not tested"), (0.0, "tab:blue", "0"), (1.0, "tab:red", "1")]:
    sel = np.isnan(labels) if np.isnan(value) else labels == value
    axes[1].scatter(pca.scores[sel, 0], pca.scores[sel, 1], s=6, c=color, label=name, alpha=0.7)
axes[1].set(title=f"PC1 vs PC2, colored by {ENDPOINT}", xlabel="PC1", ylabel="PC2")
axes[1].legend()
plt.tight_layout()
plt.show()
print("components for 90% variance:", int(np.searchsorted(np.cumsum(pca.full_variance_ratio), 0.9) + 1))

In [ ]:
nn_sim, _ = nearest_neighbor_similarity(bits)
plt.hist(nn_sim, bins=40)
plt.title("Nearest-neighbor Tanimoto similarity")
plt.xlabel("similarity to closest other molecule")
plt.show()

print("mean NN similarity:", round(float(nn_sim.mean()), 3))
print("internal diversity (1 - Tanimoto):", round(internal_diversity(bits), 3))
stats = scaffold_stats(smiles)
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in stats.items() if k != "top_scaffolds"})

## 3. Optimizers
Multitask logistic regression on the fingerprint bits (a convex model, so differences come from the optimizer). Full-batch training, masked BCE. The curves compare optimizer behavior, not model quality.

In [ ]:
import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score, roc_auc_score

from toxion.optim.optimizers import SGD, RMSprop, Adam, AdamW
from toxion.optim.schedules import make_scheduler, warmup_cosine_factor
from toxion.training.advanced_losses import (
    UncertaintyWeightedLoss, compute_pos_weights, masked_focal_loss, masked_weighted_bce,
)

N_STEPS = 200
rng = np.random.default_rng(0)
order = rng.permutation(len(df))
n_train, n_val = int(0.8 * len(df)), int(0.1 * len(df))
tr, va, te = order[:n_train], order[n_train:n_train + n_val], order[n_train + n_val:]

X = torch.tensor(bits, dtype=torch.float32)
Y = torch.tensor(df[TOX21_TASKS].to_numpy(dtype=np.float32))
Xtr, Ytr, Xva, Yva, Xte, Yte = X[tr], Y[tr], X[va], Y[va], X[te], Y[te]

def fit(make_opt, steps=N_STEPS, loss_fn=masked_weighted_bce, schedule=None, extra_params=()):
    torch.manual_seed(0)
    model = nn.Linear(N_BITS, len(TOX21_TASKS))
    opt = make_opt(list(model.parameters()) + list(extra_params))
    sched = make_scheduler(opt, schedule) if schedule else None
    hist = {"train": [], "val": []}
    for _ in range(steps):
        opt.zero_grad()
        loss = loss_fn(model(Xtr), Ytr)
        loss.backward()
        opt.step()
        if sched:
            sched.step()
        with torch.no_grad():
            hist["val"].append(masked_weighted_bce(model(Xva), Yva).item())
        hist["train"].append(loss.item())
    return model, hist

runs = {
    "SGD lr=0.02, momentum 0.9": lambda p: SGD(p, lr=0.02, momentum=0.9),
    "SGD Nesterov": lambda p: SGD(p, lr=0.02, momentum=0.9, nesterov=True),
    "RMSprop lr=1e-2": lambda p: RMSprop(p, lr=1e-2),
    "Adam lr=1e-2": lambda p: Adam(p, lr=1e-2),
    "AdamW lr=1e-2, wd=1e-2": lambda p: AdamW(p, lr=1e-2, weight_decay=1e-2),
}
histories = {name: fit(make)[1] for name, make in runs.items()}
histories["Adam + warmup/cosine"] = fit(
    lambda p: Adam(p, lr=1e-2),
    schedule=lambda s: warmup_cosine_factor(s, warmup_steps=10, total_steps=N_STEPS, min_factor=0.05),
)[1]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, h in histories.items():
    axes[0].plot(h["train"], label=name)
    axes[1].plot(h["val"], label=name)
axes[0].set(title="train loss (masked BCE)", xlabel="step", yscale="log")
axes[1].set(title="validation loss (random split, demo only)", xlabel="step")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
pd.DataFrame({name: {"final train": h["train"][-1], "best val": min(h["val"]), "final val": h["val"][-1]}
              for name, h in histories.items()}).T.round(4)

## 4. Imbalance-aware and multitask losses
Same model and optimizer (Adam), four losses: plain masked BCE, positive-class weighting (n_neg / n_pos per endpoint), focal loss, and uncertainty-weighted multitask loss with learned per-task weights.

In [ ]:
def masked_scores(y, logits):
    roc, pr = [], []
    for t in range(y.shape[1]):
        keep = ~np.isnan(y[:, t])
        if len(np.unique(y[keep, t])) < 2:
            continue
        roc.append(roc_auc_score(y[keep, t], logits[keep, t]))
        pr.append(average_precision_score(y[keep, t], logits[keep, t]))
    return float(np.mean(roc)), float(np.mean(pr))

pos_w = torch.tensor(compute_pos_weights(Ytr.numpy()))
uw = UncertaintyWeightedLoss(len(TOX21_TASKS))
adam = lambda p: Adam(p, lr=1e-2)

models = {
    "BCE": fit(adam)[0],
    "pos-weighted BCE": fit(adam, loss_fn=lambda z, y: masked_weighted_bce(z, y, pos_w))[0],
    "focal (gamma=2, alpha=0.25)": fit(adam, loss_fn=lambda z, y: masked_focal_loss(z, y, 2.0, 0.25))[0],
    "uncertainty-weighted": fit(adam, loss_fn=uw, extra_params=uw.parameters())[0],
}
rows = {}
with torch.no_grad():
    for name, m in models.items():
        roc, pr = masked_scores(Yva.numpy(), m(Xva).numpy())
        rows[name] = {"val mean ROC-AUC": roc, "val mean PR-AUC": pr}
pd.DataFrame(rows).T.round(3)

In [ ]:
pd.Series(uw.task_weights().numpy(), index=TOX21_TASKS, name="learned task weight exp(-s)").round(3)

## 5. Calibration
Per-endpoint temperature scaling fit on the validation set, evaluated on the test set. ROC-AUC does not change under temperature scaling. A loss that deliberately shifts probabilities (positive weighting) cannot be fully fixed by a temperature alone, which shows up in the ECE columns.

In [ ]:
from toxion.eval.calibration import (
    apply_temperature, expected_calibration_error, fit_temperature_per_task, reliability_curve, sigmoid,
)

def mean_ece(y, p):
    values = []
    for t in range(y.shape[1]):
        keep = ~np.isnan(y[:, t])
        if keep.sum() >= 20:
            values.append(expected_calibration_error(y[:, t], p[:, t]))
    return float(np.mean(values))

rows, curves = {}, {}
with torch.no_grad():
    for name, m in models.items():
        z_va, z_te = m(Xva).numpy(), m(Xte).numpy()
        T = fit_temperature_per_task(z_va, Yva.numpy())
        before = mean_ece(Yte.numpy(), sigmoid(z_te))
        after = mean_ece(Yte.numpy(), sigmoid(apply_temperature(z_te, T)))
        rows[name] = {"test ECE before": before, "test ECE after T-scaling": after, "median T": float(np.median(T))}
        curves[name] = (z_te, T)
pd.DataFrame(rows).T.round(3)

In [ ]:
name = "BCE"
z_te, T = curves[name]
t = TOX21_TASKS.index(ENDPOINT)
keep = ~np.isnan(Yte.numpy()[:, t])
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot([0, 1], [0, 1], "k--", label="perfect")
for label, logits in [("before", z_te[:, t]), ("after T-scaling", z_te[:, t] / T[t])]:
    conf, freq, counts = reliability_curve(Yte.numpy()[:, t], sigmoid(logits), n_bins=8)
    ax.plot(conf, freq, "o-", label=label)
ax.set(title=f"Reliability, {ENDPOINT} ({name})", xlabel="mean predicted probability", ylabel="observed frequency")
ax.legend()
plt.show()

## Notes
- Random split, demo only. Real results need a scaffold split and five seeds.
- Temperature scaling is fit on validation data; reporting ECE on the same data would be optimistic.
- ECE depends on the binning; with few positives, bins are noisy.